### Chien luoc tren SSI (1d): 
### Buy: 0 > MACD > Signal , Close > MA20
### Sell: 0 < MACD < Signal , Close < MA20

Ngắn gọn: hợp lý về ý tưởng sớm-bắt-trend, nhưng nên siết lại trigger và bổ sung thoát lệnh/rủi ro.
Gợi ý chỉnh nhanh (SSI, 1D):
Điều kiện vào lệnh (rõ ràng hơn):
Buy: MACD cắt lên Signal trong ≤3 nến gần nhất AND MACD < 0 tại lúc cắt AND Close > EMA20 AND dốc EMA20 > 0.
Sell: MACD cắt xuống Signal trong ≤3 nến AND MACD > 0 tại lúc cắt AND Close < EMA20 AND dốc EMA20 < 0.
Tránh dùng bất đẳng thức tĩnh “0 > MACD > Signal”; dùng “cross” sẽ bám sát tín hiệu động hơn và giảm nhiễu.
Lọc nhiễu tùy chọn:
RSI 14 > 40 cho Buy, < 60 cho Sell (lọc sideway cực đoan).
Volume phiên vào ≥ SMA20 volume.
Quản trị rủi ro/thoát:
Stoploss = 1.5–2.0 × ATR14; TP mục tiêu 2.5–3.0 × ATR, bật trailing 1.5 × ATR khi lãi > 2 × ATR.
Thoát khi Close cắt ngược EMA20 hoặc có tín hiệu MACD đảo chiều đối ứng.
Time-stop: thoát nếu sau 20 nến chưa đạt +1 × ATR.
Khuyến nghị: backtest nhanh hai phiên bản
A) như bạn đang viết (MACD cải thiện dưới 0/ trên 0).
B) bản “cross” ở trên + slope EMA20.
So sánh PF, MDD, win rate, expectancy; chọn cái nào ổn định hơn qua nhiều giai đoạn.

# Load data SSI: TradeSSI.py

In [1]:
import sys
sys.path.append("../Common")
import TradeSSI


In [2]:
# import pandas as pd
# from ssi_fc_data import fc_md_client , model
# import config
# client = fc_md_client.MarketDataClient(config)

# symbol = "VCB"
# from_date = "01/10/2025"
# to_date = "15/10/2025"

# data = client.daily_ohlc(config, model.daily_ohlc(symbol, from_date, to_date, 1, 100, True))
# data_list = data['data']
# data = pd.DataFrame(data_list)

# data

# Process Data

In [ ]:
# 1. Load data
# 2. Process Data => Buy/ Sell Signal
# 3. Save data to Redis
import talib
import pandas as pd

def scan_market():
	symbol = "VCB"
	from_date = "15/07/2025"
	to_date = "15/10/2025"
	data = TradeSSI.TradeSSI.get_daily_OHLC_Data(symbol, from_date, to_date)

	data["Close"] = pd.to_numeric(data["Close"])
	
	data["MA20"] = talib.SMA(data["Close"], timeperiod=20)
	data['macd'], data['macdsignal'], data['macdhist'] = talib.MACD(data['Close'], fastperiod=12, slowperiod=26, signalperiod=9)

	### Buy: 0 > MACD > Signal , Close > MA20
	### Sell: 0 < MACD < Signal , Close < MA20

	data['Buy_Signal'] = (data['macd'] > data['macdsignal']) & (data['macd'] < 0) & (data['Close'] > data['MA20'])
	data['Sell_Signal'] = (data['macd'] < data['macdsignal']) & (data['macd'] > 0) & (data['Close'] < data['MA20'])

	last_record = data.iloc(-1)


# Auto Schedule/ While True

In [ ]:
import schedule
import time

schedule.every(1).day.at("22:13").do(scan_market)

while True:
	schedule.run_pending()
	# time.sleep(1)   

   Symbol Market TradingDate  Time   Open   High    Low  Close   Volume  \
0     VCB   HOSE  15/07/2025  None  61951  62149  60958  60958  6315400   
1     VCB   HOSE  16/07/2025  None  60958  61653  60660  61355  7603200   
2     VCB   HOSE  17/07/2025  None  61355  62249  61355  61752  9992700   
3     VCB   HOSE  18/07/2025  None  61851  62149  61355  61454  6915300   
4     VCB   HOSE  21/07/2025  None  61752  62249  60859  60859  6138200   
..    ...    ...         ...   ...    ...    ...    ...    ...      ...   
60    VCB   HOSE  09/10/2025  None  64500  65200  63800  63800  5734200   
61    VCB   HOSE  10/10/2025  None  63900  64400  63600  64200  4985600   
62    VCB   HOSE  13/10/2025  None  63400  63800  62800  63100  7992500   
63    VCB   HOSE  14/10/2025  None  63500  64000  62900  63100  7008900   
64    VCB   HOSE  15/10/2025  None  63100  63400  62500  62500  5218800   

           Value      MA20        macd  macdsignal    macdhist  Buy_Signal  \
0   391249840000     